# 13. RoPE 长上下文缩放：怎样从旋转相位做到可验证的远距离检索？

## 面试回答主线

RoPE 把每两个通道看成一个二维平面，并按“位置 × 频率”旋转 query 与 key。它的重要性质是两者内积由相对位置决定，而不是由绝对位置直接决定。模型只在 128 token 内训练，却直接读取 1024 token 时，未见过的大相位会让远处语义 key 的分数周期性翻转。Position Interpolation 将部署位置压回训练坐标范围；它不是免费扩窗，而是一项必须与模型权重、KV cache 和推理配置共同版本化的合同。面试时我会先在同一批远距离 needle 上比较未缩放基线与插值方案，再检查相位、排名和 cache 配置。最后还要说明：这个小实验验证的是几何机制，不代表真实模型已经获得长文本理解能力。

## 1. 真实案例：长工单里检索早期承诺

我们构造 6 条客服工单，每条 query 都在长上下文尾部询问前文中的关键承诺。每个样本包含目标 key、一个语义相近的近期套话和一个无关 key；目标位置跨度覆盖 160～820 token。训练长度固定为 128、部署长度为 1024，因此插值比例是 1/8。下面先预览业务字段和相对距离，而不是从随机张量开始。

In [1]:
from pprint import pprint  # 导入结构化打印工具以展示真实工单输入
train_length = 128  # 记录模型训练时见过的最大位置范围
deploy_length = 1024  # 记录当前服务需要承载的上下文长度
cases = [{"id": "T01", "query": "退款承诺是哪一天？", "query_pos": 180, "target": "坐席承诺周五退款", "target_pos": 20}, {"id": "T02", "query": "补寄使用哪家快递？", "query_pos": 300, "target": "补寄将使用顺丰", "target_pos": 40}, {"id": "T03", "query": "赔付金额是多少？", "query_pos": 480, "target": "确认赔付八十元", "target_pos": 100}, {"id": "T04", "query": "安装师傅几点到？", "query_pos": 700, "target": "师傅约在下午三点", "target_pos": 120}, {"id": "T05", "query": "换货单号是什么？", "query_pos": 920, "target": "换货单号是 EX2048", "target_pos": 160}, {"id": "T06", "query": "发票抬头写什么？", "query_pos": 1020, "target": "抬头写星河科技", "target_pos": 200}]  # 定义六条具有明确答案位置的长工单
preview = [{"工单": item["id"], "问题": item["query"], "目标事实": item["target"], "相对距离": item["query_pos"] - item["target_pos"]} for item in cases]  # 汇总学习者需要观察的业务字段
print("真实案例输入预览：")  # 输出本节标题帮助定位结果
pprint(preview, sort_dicts=False)  # 展示六条工单及其远距离跨度

真实案例输入预览：
[{'工单': 'T01', '问题': '退款承诺是哪一天？', '目标事实': '坐席承诺周五退款', '相对距离': 160},
 {'工单': 'T02', '问题': '补寄使用哪家快递？', '目标事实': '补寄将使用顺丰', '相对距离': 260},
 {'工单': 'T03', '问题': '赔付金额是多少？', '目标事实': '确认赔付八十元', '相对距离': 380},
 {'工单': 'T04', '问题': '安装师傅几点到？', '目标事实': '师傅约在下午三点', '相对距离': 580},
 {'工单': 'T05', '问题': '换货单号是什么？', '目标事实': '换货单号是 EX2048', '相对距离': 760},
 {'工单': 'T06', '问题': '发票抬头写什么？', '目标事实': '抬头写星河科技', '相对距离': 820}]


## 2. Baseline（基线）：把 1024 位置直接送进训练长度为 128 的 RoPE

为隔离位置效应，目标 key 与 query 使用相同语义方向；近期套话只有 0.86 的语义强度。未缩放基线直接用部署位置计算相位，远距离目标的余弦分数会下降甚至变负。它与后面的方案使用完全相同的样本、候选和频率，差别只有位置坐标合同。

In [2]:
import math  # 导入三角函数以手写二维 RoPE 旋转
frequency = 0.002  # 选择能直观看到长距离相位漂移的低频通道
def rotate(vector, position, scale):  # 定义二维向量的 RoPE 旋转函数
    angle = position * scale * frequency  # 计算当前位置在指定缩放合同下的旋转相位
    cosine = math.cos(angle)  # 计算旋转矩阵的余弦项
    sine = math.sin(angle)  # 计算旋转矩阵的正弦项
    return (vector[0] * cosine - vector[1] * sine, vector[0] * sine + vector[1] * cosine)  # 返回手写矩阵乘法后的二维向量
def dot(left, right):  # 定义注意力打分所需的二维内积
    return left[0] * right[0] + left[1] * right[1]  # 返回两个旋转向量的相似度
def retrieve(item, scale):  # 在一个工单的三个候选 key 中执行检索
    query_vector = rotate((1.0, 0.0), item["query_pos"], scale)  # 按 query 的全局位置旋转语义向量
    candidates = [("目标事实", (1.0, 0.0), item["target_pos"]), ("近期套话", (0.86, 0.0), item["query_pos"] - 1), ("无关状态", (0.0, 0.65), item["query_pos"] - 3)]  # 构造同一工单内可比较的候选
    scores = {name: round(dot(query_vector, rotate(vector, position, scale)), 4) for name, vector, position in candidates}  # 计算每个候选的旋转内积
    winner = max(scores, key=scores.get)  # 选择分数最高的候选作为检索结果
    return winner, scores  # 同时返回最终选择和可解释中间分数
baseline_rows = []  # 收集未缩放方案的逐工单结果
for item in cases:  # 逐条运行相同的长工单输入
    winner, scores = retrieve(item, 1.0)  # 用未缩放的绝对位置执行基线检索
    baseline_rows.append({"工单": item["id"], "目标分": scores["目标事实"], "套话分": scores["近期套话"], "命中": winner == "目标事实"})  # 保存便于横向比较的指标
print("未缩放基线的逐样本结果：")  # 标注当前输出属于基线
pprint(baseline_rows, sort_dicts=False)  # 展示相位漂移导致的真实错检

未缩放基线的逐样本结果：
[{'工单': 'T01', '目标分': 0.9492, '套话分': 0.86, '命中': True},
 {'工单': 'T02', '目标分': 0.8678, '套话分': 0.86, '命中': True},
 {'工单': 'T03', '目标分': 0.7248, '套话分': 0.86, '命中': False},
 {'工单': 'T04', '目标分': 0.3993, '套话分': 0.86, '命中': False},
 {'工单': 'T05', '目标分': 0.0508, '套话分': 0.86, '命中': False},
 {'工单': 'T06', '目标分': -0.0691, '套话分': 0.86, '命中': False}]


## 3. 手写核心算法：Position Interpolation 与相位中间量

插值比例为 `train_length / deploy_length`，部署位置 1020 会被映射到 127.5，仍处于训练相位范围。下面不调用任何 RoPE 封装，而是直接复用手写旋转矩阵。除了最终命中，还打印每条样本的原始相对相位与插值相对相位，说明改变来自哪里。

In [3]:
position_scale = train_length / deploy_length  # 计算把部署坐标压回训练坐标的插值比例
phase_rows = []  # 收集每个样本的相位变化以解释算法机制
for item in cases:  # 遍历所有远距离检索样本
    relative_distance = item["query_pos"] - item["target_pos"]  # 计算 query 与目标 key 的相对位置
    raw_phase = relative_distance * frequency  # 计算未缩放方案看到的相对旋转相位
    scaled_phase = relative_distance * position_scale * frequency  # 计算插值后落入训练范围的相对相位
    phase_rows.append({"工单": item["id"], "原始相位": round(raw_phase, 3), "插值相位": round(scaled_phase, 3), "目标余弦": round(math.cos(scaled_phase), 4)})  # 保存可审计的几何中间量
print(f"位置插值比例：{position_scale:.3f}")  # 输出部署长度与训练长度形成的缩放比例
pprint(phase_rows, sort_dicts=False)  # 展示每个样本的相位如何被压回已训练区域

位置插值比例：0.125
[{'工单': 'T01', '原始相位': 0.32, '插值相位': 0.04, '目标余弦': 0.9992},
 {'工单': 'T02', '原始相位': 0.52, '插值相位': 0.065, '目标余弦': 0.9979},
 {'工单': 'T03', '原始相位': 0.76, '插值相位': 0.095, '目标余弦': 0.9955},
 {'工单': 'T04', '原始相位': 1.16, '插值相位': 0.145, '目标余弦': 0.9895},
 {'工单': 'T05', '原始相位': 1.52, '插值相位': 0.19, '目标余弦': 0.982},
 {'工单': 'T06', '原始相位': 1.64, '插值相位': 0.205, '目标余弦': 0.9791}]


## 4. 在同一批工单上比较两种方案

现在只把 `scale` 从 1.0 改为 0.125，其他输入完全不变。逐样本表格同时给出基线选择、插值选择和目标分数，避免只报告平均准确率。

In [4]:
comparison = []  # 收集同数据上的基线与插值方案对照
for item in cases:  # 对每个工单执行两套坐标合同
    baseline_winner, baseline_scores = retrieve(item, 1.0)  # 获取未缩放方案的选择与分数
    scaled_winner, scaled_scores = retrieve(item, position_scale)  # 获取位置插值方案的选择与分数
    comparison.append({"工单": item["id"], "基线选择": baseline_winner, "插值选择": scaled_winner, "基线目标分": baseline_scores["目标事实"], "插值目标分": scaled_scores["目标事实"]})  # 保存逐样本可解释结果
baseline_hits = sum(row["基线选择"] == "目标事实" for row in comparison)  # 统计基线正确检索数量
scaled_hits = sum(row["插值选择"] == "目标事实" for row in comparison)  # 统计插值方案正确检索数量
print("同数据逐样本对照：")  # 输出结果表标题
pprint(comparison, sort_dicts=False)  # 展示每个业务样本的选择变化
print(f"命中数：未缩放 {baseline_hits}/{len(cases)}，位置插值 {scaled_hits}/{len(cases)}")  # 汇总两种方案的可比较命中率

同数据逐样本对照：
[{'工单': 'T01',
  '基线选择': '目标事实',
  '插值选择': '目标事实',
  '基线目标分': 0.9492,
  '插值目标分': 0.9992},
 {'工单': 'T02',
  '基线选择': '目标事实',
  '插值选择': '目标事实',
  '基线目标分': 0.8678,
  '插值目标分': 0.9979},
 {'工单': 'T03',
  '基线选择': '近期套话',
  '插值选择': '目标事实',
  '基线目标分': 0.7248,
  '插值目标分': 0.9955},
 {'工单': 'T04',
  '基线选择': '近期套话',
  '插值选择': '目标事实',
  '基线目标分': 0.3993,
  '插值目标分': 0.9895},
 {'工单': 'T05', '基线选择': '近期套话', '插值选择': '目标事实', '基线目标分': 0.0508, '插值目标分': 0.982},
 {'工单': 'T06',
  '基线选择': '近期套话',
  '插值选择': '目标事实',
  '基线目标分': -0.0691,
  '插值目标分': 0.9791}]
命中数：未缩放 2/6，位置插值 6/6


## 5. 结果解读：几何修复不等于能力凭空增加

在这个受控案例中，未缩放基线随着距离变长逐步被近期套话超过，位置插值则恢复了 6 条目标事实的排名。改进来自相对相位不再越过训练范围，而不是把目标答案偷偷加权。真实 LLM 还会叠加多频率、多 head、注意力稀释和训练分布，因此应在 needle、长文问答、困惑度及短上下文回归集上共同验收。下面输出距离分桶，确认提升确实集中在长距离样本。

In [5]:
distance_report = []  # 构造距离与收益之间的解释表
for item, row in zip(cases, comparison):  # 对齐原始样本和比较结果
    distance = item["query_pos"] - item["target_pos"]  # 读取当前样本的相对距离
    gain = float(row["插值选择"] == "目标事实") - float(row["基线选择"] == "目标事实")  # 计算当前样本是否因插值转为正确
    distance_report.append({"工单": item["id"], "距离": distance, "命中收益": gain})  # 保存逐样本收益而非只留均值
print("距离与检索收益：")  # 标记结果解读所用数据
pprint(distance_report, sort_dicts=False)  # 展示长距离样本的收益分布

距离与检索收益：
[{'工单': 'T01', '距离': 160, '命中收益': 0.0},
 {'工单': 'T02', '距离': 260, '命中收益': 0.0},
 {'工单': 'T03', '距离': 380, '命中收益': 1.0},
 {'工单': 'T04', '距离': 580, '命中收益': 1.0},
 {'工单': 'T05', '距离': 760, '命中收益': 1.0},
 {'工单': 'T06', '距离': 820, '命中收益': 1.0}]


## 6. 失败案例与修正：旧 RoPE 配置的 KV cache 被新请求复用

最危险的线上错误不是公式写错，而是 key 已按旧 scale=1.0 写入 cache，query 却按新 scale=0.125 旋转。两边不再共享同一坐标系，即使向量语义完全相同，内积也会失真。修正是把 `rope_scale`、base、维度和模型版本写进 cache key，指纹不一致就拒绝复用并重建。

In [6]:
cache_case = cases[-1]  # 选择距离最长的工单复现配置混用事故
cached_position = 900  # 选择长上下文尾部的一条近期证据作为缓存 key
cached_key_old = rotate((1.0, 0.0), cached_position, 1.0)  # 模拟旧配置已经写入缓存的目标 key
query_new = rotate((1.0, 0.0), cache_case["query_pos"], position_scale)  # 模拟新配置生成的 query
mixed_score = dot(query_new, cached_key_old)  # 计算坐标合同不一致时的错误注意力分数
cached_key_new = rotate((1.0, 0.0), cached_position, position_scale)  # 按新配置重建目标 key 缓存
fixed_score = dot(query_new, cached_key_new)  # 计算 query 与 key 合同一致后的正确分数
old_fingerprint = f"rope-scale=1.0|length={deploy_length}"  # 生成旧缓存的 RoPE 配置指纹
new_fingerprint = f"rope-scale={position_scale}|length={deploy_length}"  # 生成当前请求的 RoPE 配置指纹
cache_reusable = old_fingerprint == new_fingerprint  # 用严格指纹判断缓存能否安全复用
print({"失败复现_混合分数": round(mixed_score, 4), "修正后分数": round(fixed_score, 4), "允许复用旧缓存": cache_reusable})  # 展示事故影响和配置门禁结果

{'失败复现_混合分数': 0.0258, '修正后分数': 0.9996, '允许复用旧缓存': False}


## 7. 生产差距与最小回归检查

生产实现还要覆盖所有 RoPE 频率、head 和 dtype，并确认训练框架与推理 kernel 对缩放公式的定义完全一致。需要把模型版本、tokenizer、RoPE 参数和 KV cache 指纹作为一个发布单元；上线前同时跑短上下文无回退与长上下文收益测试。Position Interpolation 可能牺牲局部位置分辨率，必要时还要比较 NTK-aware、YaRN 或继续预训练。最后的断言只守住本实验已经展示的关键不变量，不替代前面的结果表。

In [7]:
assert len(cases) >= 5  # 确认真实案例数量满足逐样本比较要求
assert scaled_hits > baseline_hits  # 确认位置插值在同一批长工单上带来可见收益
assert scaled_hits == len(cases)  # 确认受控案例中的目标事实都被正确找回
assert fixed_score > mixed_score  # 确认重建一致配置的缓存修复了分数失真
assert cache_reusable is False  # 确认配置指纹门禁拒绝复用旧 RoPE 缓存
print("回归检查通过：缩放收益、逐样本命中和 KV cache 配置门禁均已验证。")  # 输出简洁的最终验收结论

回归检查通过：缩放收益、逐样本命中和 KV cache 配置门禁均已验证。
